# Autoencoders Accuracy

In [ ]:
import sys
with open("./../../../PATHS.txt") as file:
  paths = file.read().splitlines()
sys.path.extend(paths)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
dim_ranges = {
  "interior": {
    "latent_dim": np.arange(8,21,2),
    "row_nonzero": np.arange(6,13,2)
  },
  "interface": {
    "latent_dim": np.arange(6,16,3),
    "row_nonzero": np.arange(4,11,2)
  },
  "port": {
    "latent_dim": np.arange(4,11,2),
    "row_nonzero": np.arange(4,9,1)
  }
}
elements = ["interior", "port"]
path_to_nets = "/g/g92/zanardi1/Workspace/Codes/DD-NM-ROM/run/unsteady/dset.2by2/nets/run.02/"
path_to_figs = "/g/g92/zanardi1/Workspace/Codes/DD-NM-ROM/run/unsteady/dset.2by2/figures/global/nets/run.02/"

In [ ]:
suffix = {
  "port": {
    "horiz": "orient_horiz_size_96_ports_0_3",
    "vert": "orient_vert_size_96_ports_4_7"
  },
  "interior": "subs_0_1_2_3",
  "interface": "subs_0_1_2_3"
}
title = {
  "port": {
    "horiz": "Horizontal Ports",
    "vert": "Vertical Ports"
  },
  "interior": "Interior",
  "interface": "Interface"
}

In [ ]:
def read_accuracy(element, dims, path_to_nets, suffix):
  accuracy = []
  for (ld, rnz) in dims:
    filename = path_to_nets + f"/{element}_ld_{ld}_rnz_{rnz}/merged/{suffix}/refine/training/history.csv"
    accuracy.append(pd.read_csv(filename)["val_loss"].min())
  return np.array(accuracy)

def plot_accuracy(dim_ranges, accuracy, title, filename):
  fig, ax = plt.subplots()
  # Set x-axis
  x = np.arange(len(dim_ranges["row_nonzero"]))
  x_labels = dim_ranges["row_nonzero"].astype(str)
  ax.set_xticks(x, labels=x_labels)
  ax.set_xlabel("Nonzero Rows - Layer Density")
  # Set y-axis
  y = np.arange(len(dim_ranges["latent_dim"]))
  y_labels = dim_ranges["latent_dim"].astype(str)
  ax.set_yticks(y, labels=y_labels)
  ax.set_ylabel("Latent Dimension")
  # Set z-axis
  z = accuracy.reshape(len(y), len(x))
  im = ax.imshow(z)
  cbar = fig.colorbar(im, orientation='vertical', label="MSE Validation Loss")
  cbar.formatter.set_powerlimits((0, 0))
  # Set 
  plt.title(title)
  plt.savefig(filename, bbox_inches="tight", pad_inches=0.1)
  plt.show()

In [ ]:
os.makedirs(path_to_figs, exist_ok=True)
for element in elements:
  dims = np.meshgrid(
    dim_ranges[element]["latent_dim"],
    dim_ranges[element]["row_nonzero"]
  )
  dims = np.array(dims).T.reshape(-1,2)
  if (element == "port"):
    for orient in ("horiz", "vert"):
      isuffix = element + "/" + suffix[element][orient]
      accuracy = read_accuracy(element, dims, path_to_nets, isuffix)
      filename = path_to_figs + f"/acc_{element}_{orient}.png"
      plot_accuracy(dim_ranges[element], accuracy, title[element][orient], filename)
  else:
    isuffix = element + "/" + suffix[element]
    accuracy = read_accuracy(element, dims, path_to_nets, isuffix)
    filename = path_to_figs + f"/acc_{element}.png"
    plot_accuracy(dim_ranges[element], accuracy, title[element], filename)